# Notebook 8: Uplift Modeling - Heterogeneous Treatment Effects

**Finding Who Benefits Most from the Email Campaign**

## Overview

In typical A/B testing, we calculate the **average** effect of the email: "On average, customers who receive Mens Email have X% higher conversion rate." But here's the catch: not everyone responds the same way!

**Uplift modeling** asks: "For which individual customers is the email beneficial?" Some customers might be more likely to buy anyway (with or without email), while others might be genuinely persuaded by it. This notebook demonstrates causal machine learning techniques to identify customers with the highest individual treatment effects.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import plotly.graph_objects as go
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Create output directory
os.makedirs('../data/outputs/nb08', exist_ok=True)


In [ ]:
# Load data
df = pd.read_csv('../data/outputs/nb01/nb01_hillstrom_clean.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nTreatment groups:")
print(df['segment'].value_counts())

# For this analysis, focus on Mens Email vs No Email
df_analysis = df[df['segment'].isin(['Mens E-Mail', 'No E-Mail'])].copy()
df_analysis['treatment'] = (df_analysis['segment'] == 'Mens E-Mail').astype(int)

print(f"\nAnalysis sample size: {len(df_analysis)}")
print(f"Conversion rate by group:")
print(df_analysis.groupby('segment')['conversion'].mean())

## Concept: Average Treatment Effect vs Heterogeneous Treatment Effects

### Average Treatment Effect (ATE)
The average difference between treatment and control groups across all customers.
- Easy to calculate
- Good for policy decisions ("Should we send emails to everyone?")
- But misses variation within groups

### Heterogeneous Treatment Effects (HTE) / Uplift
The individual treatment effect for each customer.
- Depends on customer characteristics
- Useful for targeted decisions ("Who should we email?")
- Requires more sophisticated modeling

### Four Customer Types (Qini Classification)

```
                    | Receives Email | No Email
Buys Anyway         |   ✓ (Spend)   | ✓(Lost cause)
Won't Buy           |   ✗ (Wasted)  | ✗
Persuadable/Uplift  |   ✓ (Benefit) | ✗ (Your targets!)
Sleeping Dogs       |   ? (Backlash)| Could be fine
```

**Our Goal:** Identify "Persuadables" - customers more likely to convert BECAUSE of the email.

### Why This Matters

1. **Targeting**: Send emails only to high-uplift customers
2. **Cost Reduction**: Avoid wasting emails on lost causes
3. **Efficiency**: Focus budget on customers who respond
4. **Personalization**: Different messaging for different types

## Two-Model Approach (T-Learner)

**T-Learner** trains separate models for treatment and control groups.

### Algorithm
1. **Model 1 (Control)**: Train on customers who didn't receive email
   - Learns: P(conversion | features, no email)
2. **Model 2 (Treatment)**: Train on customers who received email
   - Learns: P(conversion | features, email)
3. **Uplift**: For each customer, calculate
   - Uplift = P(conversion | features, email) - P(conversion | features, no email)
4. **Ranking**: Rank customers by uplift score

### Advantages
- Straightforward interpretation
- Can use different model types for each group
- Works well when treatment effect varies with features

In [ ]:
# Prepare features for uplift modeling
# Encode categorical variables
df_model = df_analysis.copy()

# Encode channel (if categorical)
if df_model['channel'].dtype == 'object':
    le_channel = LabelEncoder()
    df_model['channel_encoded'] = le_channel.fit_transform(df_model['channel'])
else:
    df_model['channel_encoded'] = df_model['channel']

# Encode zip_code if categorical (just use numeric codes)
if df_model['zip_code'].dtype == 'object':
    le_zip = LabelEncoder()
    df_model['zip_encoded'] = le_zip.fit_transform(df_model['zip_code'])
else:
    df_model['zip_encoded'] = df_model['zip_code']

# Select features for modeling
feature_cols = ['recency', 'history', 'mens', 'womens', 'zip_encoded', 'newbie', 'channel_encoded']
X = df_model[feature_cols]
y = df_model['conversion'].astype(int)
W = df_model['treatment']

print(f"Features: {feature_cols}")
print(f"Sample size: {len(X)}")
print(f"Treatment rate: {W.mean():.2%}")
print(f"Conversion rate: {y.mean():.2%}")
print(f"\nFeature statistics:")
print(X.describe())

In [ ]:
# T-Learner: Train separate models
# Split data by treatment status
X_control = X[W == 0]
y_control = y[W == 0]
X_treatment = X[W == 1]
y_treatment = y[W == 1]

print(f"Control sample: {len(X_control)}")
print(f"Treatment sample: {len(X_treatment)}")
print(f"Control conversion: {y_control.mean():.2%}")
print(f"Treatment conversion: {y_treatment.mean():.2%}")

# Train logistic regression models (simpler, interpretable)
model_control = LogisticRegression(max_iter=1000, random_state=42)
model_control.fit(X_control, y_control)

model_treatment = LogisticRegression(max_iter=1000, random_state=42)
model_treatment.fit(X_treatment, y_treatment)

# Predict for all customers
pred_prob_control = model_control.predict_proba(X)[:, 1]  # P(Y=1 | X, no email)
pred_prob_treatment = model_treatment.predict_proba(X)[:, 1]  # P(Y=1 | X, email)

# Calculate uplift
df_model['uplift_tlearner'] = pred_prob_treatment - pred_prob_control

print(f"\n=== T-Learner Uplift Results ===")
print(f"Mean predicted control conversion: {pred_prob_control.mean():.4f}")
print(f"Mean predicted treatment conversion: {pred_prob_treatment.mean():.4f}")
print(f"Average uplift: {df_model['uplift_tlearner'].mean():.4f}")
print(f"\nUplift distribution:")
print(df_model['uplift_tlearner'].describe())
print(f"\nPositive uplift (good targets): {(df_model['uplift_tlearner'] > 0).mean():.1%}")

## Single-Model Approach (S-Learner)

**S-Learner** trains one model with treatment as a feature.

### Algorithm
1. Train one model: P(Y | X, Treatment)
2. For each customer, predict twice:
   - P(Y | X, Treatment=1) with email
   - P(Y | X, Treatment=0) without email
3. Uplift = Difference

### Advantages
- Simpler training
- Regularization helps with variance
- Good baseline

### Disadvantages  
- May struggle to capture interaction effects
- Treatment coefficient alone doesn't give individual effects

In [ ]:
# S-Learner: Single model with treatment as feature
X_with_treatment = X.copy()
X_with_treatment['treatment'] = W.values

model_s = LogisticRegression(max_iter=1000, random_state=42)
model_s.fit(X_with_treatment, y)

# Predict with treatment=1 and treatment=0
X_treated = X_with_treatment.copy()
X_treated['treatment'] = 1
X_control_s = X_with_treatment.copy()
X_control_s['treatment'] = 0

pred_treated_s = model_s.predict_proba(X_treated)[:, 1]
pred_control_s = model_s.predict_proba(X_control_s)[:, 1]

df_model['uplift_slearner'] = pred_treated_s - pred_control_s

print("=== S-Learner Uplift Results ===")
print(f"Average uplift: {df_model['uplift_slearner'].mean():.4f}")
print(f"Positive uplift: {(df_model['uplift_slearner'] > 0).mean():.1%}")

# Compare T-Learner vs S-Learner
correlation = np.corrcoef(df_model['uplift_tlearner'], df_model['uplift_slearner'])[0, 1]
print(f"\nCorrelation between T-Learner and S-Learner: {correlation:.4f}")
print(f"\nBoth methods identify similar customers, but with different magnitudes."
      f"\nT-Learner typically more flexible for heterogeneous effects.")

## Class Variable Transformation (Jaskowski Approach)

An alternative approach for binary outcomes:

1. Create transformed outcome:
   - Z = Y * W / P(W=1) - Y * (1-W) / P(W=0)
   - Where W is treatment indicator
2. Train one model predicting Z from features
3. Predicted Z is the uplift

**Intuition:** This transformation weights the outcomes by treatment probability, making the model focus on treatment response.

In [ ]:
# Class Variable Transformation approach
p_treatment = W.mean()

# Create transformed outcome Z
Z = (y.values * W.values / p_treatment - 
     y.values * (1 - W.values) / (1 - p_treatment))

# Train model on Z
model_z = LinearRegression()
model_z.fit(X, Z)

df_model['uplift_transformed'] = model_z.predict(X)

print("=== Class Variable Transformation Results ===")
print(f"Mean transformed outcome: {Z.mean():.4f}")
print(f"Average predicted uplift: {df_model['uplift_transformed'].mean():.4f}")

# Compare all three methods
print(f"\n=== Comparison of All Methods ===")
methods_corr = pd.DataFrame({
    'T-Learner': df_model['uplift_tlearner'],
    'S-Learner': df_model['uplift_slearner'],
    'Transformed': df_model['uplift_transformed']
}).corr()

print(methods_corr)

In [ ]:
# Visualize uplift distributions by method
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

methods_list = ['uplift_tlearner', 'uplift_slearner', 'uplift_transformed']
method_names = ['T-Learner', 'S-Learner', 'Transformed']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for idx, (method, name, color) in enumerate(zip(methods_list, method_names, colors)):
    ax = axes[idx]
    ax.hist(df_model[method], bins=50, color=color, alpha=0.7, edgecolor='black')
    ax.axvline(df_model[method].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df_model[method].mean():.4f}')
    ax.axvline(0, color='black', linestyle=':', linewidth=1.5, label='Zero (No effect)')
    ax.set_xlabel('Predicted Uplift')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{name} Uplift Distribution')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb08/nb08_uplift_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Uplift Evaluation: Qini Curve and AUUC

### Qini Curve
Shows cumulative incremental conversions as we target higher-uplift customers.

**x-axis:** % of population targeted (ordered by uplift, highest first)
**y-axis:** Cumulative incremental conversions vs random targeting

### Area Under Uplift Curve (AUUC)
Summary metric: area between Qini curve and random baseline.
- Higher AUUC = better model
- Can compare different uplift models

### Qini Coefficient
Normalized version of AUUC (0 to 1 scale)

In [ ]:
def calculate_qini_curve(df, uplift_col, y_col, treatment_col):
    """Calculate Qini curve for uplift model."""
    # Sort by predicted uplift (descending)
    df_sorted = df.sort_values(uplift_col, ascending=False).reset_index(drop=True)
    
    n = len(df_sorted)
    n_treatment = df_sorted[treatment_col].sum()
    n_control = n - n_treatment
    
    # For each percentile, calculate incremental lift
    percentiles = np.arange(0, n+1)
    qini = np.zeros(len(percentiles))
    
    for i, pct in enumerate(percentiles):
        if pct == 0:
            qini[i] = 0
        else:
            subset = df_sorted.iloc[:int(pct*n/100)]
            treatment_conversions = subset[subset[treatment_col] == 1][y_col].sum()
            control_conversions = subset[subset[treatment_col] == 0][y_col].sum()
            
            # Normalize by group sizes in sample
            if subset[treatment_col].sum() > 0:
                treatment_rate = treatment_conversions / subset[treatment_col].sum()
            else:
                treatment_rate = 0
            
            if (subset[treatment_col] == 0).sum() > 0:
                control_rate = control_conversions / (subset[treatment_col] == 0).sum()
            else:
                control_rate = 0
            
            # Incremental = (treatment_rate - control_rate) * n_in_sample
            qini[i] = (treatment_rate - control_rate) * len(subset)
    
    return percentiles, qini

# Calculate Qini curves for all methods
percentiles_t, qini_t = calculate_qini_curve(df_model, 'uplift_tlearner', 'conversion', 'treatment')
percentiles_s, qini_s = calculate_qini_curve(df_model, 'uplift_slearner', 'conversion', 'treatment')
percentiles_z, qini_z = calculate_qini_curve(df_model, 'uplift_transformed', 'conversion', 'treatment')

# Random baseline (target by random order)
qini_random = percentiles_t * (df_model['conversion'].mean())

print("Qini Curve calculated for all three methods")

In [ ]:
# Plot Qini curves
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(percentiles_t, qini_t, marker='o', markersize=4, label='T-Learner', linewidth=2.5, color='#1f77b4')
ax.plot(percentiles_s, qini_s, marker='s', markersize=4, label='S-Learner', linewidth=2.5, color='#ff7f0e')
ax.plot(percentiles_z, qini_z, marker='^', markersize=4, label='Transformed', linewidth=2.5, color='#2ca02c')
ax.plot(percentiles_t, qini_random, linestyle='--', label='Random (Baseline)', linewidth=2, color='gray')

ax.fill_between(percentiles_t, qini_random, qini_t, alpha=0.2, color='#1f77b4', label='AUUC (T-Learner)')
ax.set_xlabel('Percentage of Population Targeted (%) ', fontsize=12)
ax.set_ylabel('Cumulative Incremental Conversions', fontsize=12)
ax.set_title('Qini Curve: Value of Targeting High-Uplift Customers', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb08/nb08_qini_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Calculate AUUC (Area Under Uplift Curve)
auuc_t = np.trapz(qini_t - qini_random, percentiles_t) / 100
auuc_s = np.trapz(qini_s - qini_random, percentiles_s) / 100
auuc_z = np.trapz(qini_z - qini_random, percentiles_z) / 100

print(f"\n=== AUUC Results ===")
print(f"T-Learner AUUC: {auuc_t:.4f}")
print(f"S-Learner AUUC: {auuc_s:.4f}")
print(f"Transformed AUUC: {auuc_z:.4f}")
print(f"\nHigher AUUC = Better targeting ability")

## Targeting Strategy

Once we have uplift predictions, we can develop a targeting strategy:

1. **Rank all customers by predicted uplift**
2. **Target top percentiles** where uplift is highest
3. **Calculate expected lift** vs random or universal sending
4. **Identify optimal targeting depth** (sweet spot in Qini curve)

The Qini curve tells us: "If we target the top X% of customers by uplift, how many incremental conversions do we gain?"

**Business Decision:** How much margin do we need per conversion to make emailing worthwhile?

In [ ]:
# Find optimal targeting percentage
# Where does targeting top X% give best ROI?

# Assume: email cost = $0.50, conversion value = $50 (margin)
email_cost = 0.50
conversion_value = 50
baseline_conversion_rate = df_model['conversion'].mean()

# For each targeting percentile, calculate net value
targeting_pcts = np.arange(1, 101)
net_values = []
incremental_conversions = []

for pct in targeting_pcts:
    # Number of people we'd email
    n_target = len(df_model) * pct / 100
    
    # Expected additional conversions from targeting (vs random)
    expected_incremental = (qini_t[int(pct)] - qini_random[int(pct)])
    incremental_conversions.append(expected_incremental)
    
    # Net value
    net_value = (expected_incremental * conversion_value) - (n_target * email_cost)
    net_values.append(net_value)

optimal_pct = targeting_pcts[np.argmax(net_values)]
optimal_value = max(net_values)

print(f"=== Optimal Targeting Strategy ===")
print(f"Email cost: ${email_cost:.2f}")
print(f"Value per conversion: ${conversion_value:.2f}")
print(f"\nOptimal: Target top {optimal_pct}% of customers")
print(f"Expected net value: ${optimal_value:.2f} total")
print(f"Expected incremental conversions: {incremental_conversions[optimal_pct-1]:.0f}")

# Visualize targeting analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(targeting_pcts, incremental_conversions, linewidth=2.5, color='#2ca02c')
ax1.axvline(optimal_pct, color='red', linestyle='--', linewidth=2, label=f'Optimal: {optimal_pct}%')
ax1.set_xlabel('Percentage of Population Targeted (%)', fontsize=11)
ax1.set_ylabel('Incremental Conversions vs Random', fontsize=11)
ax1.set_title('Incremental Conversions by Targeting Depth', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(targeting_pcts, net_values, linewidth=2.5, color='#1f77b4')
ax2.axvline(optimal_pct, color='red', linestyle='--', linewidth=2, label=f'Optimal: {optimal_pct}%')
ax2.axhline(0, color='black', linestyle=':', linewidth=1)
ax2.set_xlabel('Percentage of Population Targeted (%)', fontsize=11)
ax2.set_ylabel('Net Value ($)', fontsize=11)
ax2.set_title('Net Campaign Value by Targeting Depth', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb08/nb08_targeting_strategy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Analyze uplift by customer decile
df_model['uplift_decile'] = pd.qcut(df_model['uplift_tlearner'], q=10, labels=False, duplicates='drop')

uplift_by_decile = df_model.groupby('uplift_decile').agg({
    'uplift_tlearner': 'mean',
    'conversion': 'mean',
    'spend': 'mean',
    'treatment': 'mean',
    'recency': 'mean',
    'history': 'mean'
}).round(4)

uplift_by_decile.columns = ['Avg Uplift', 'Conversion Rate', 'Avg Spend', 'Treatment Rate', 'Avg Recency', 'Avg History']

print("=== Uplift Analysis by Decile (Lowest to Highest) ===")
print(uplift_by_decile)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(uplift_by_decile.index, uplift_by_decile['Avg Uplift'], marker='o', linewidth=2.5, color='#1f77b4')
axes[0, 0].set_xlabel('Decile (0=Lowest, 9=Highest Uplift)')
axes[0, 0].set_ylabel('Average Predicted Uplift')
axes[0, 0].set_title('Uplift Score by Decile')
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(uplift_by_decile.index, uplift_by_decile['Conversion Rate'], marker='s', linewidth=2.5, color='#ff7f0e')
axes[0, 1].set_xlabel('Decile')
axes[0, 1].set_ylabel('Conversion Rate')
axes[0, 1].set_title('Conversion Rate by Uplift Decile')
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(uplift_by_decile.index, uplift_by_decile['Avg Spend'], marker='^', linewidth=2.5, color='#2ca02c')
axes[1, 0].set_xlabel('Decile')
axes[1, 0].set_ylabel('Average Spend ($)')
axes[1, 0].set_title('Average Spend by Uplift Decile')
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(uplift_by_decile.index, uplift_by_decile['Avg History'], marker='d', linewidth=2.5, color='#d62728')
axes[1, 1].set_xlabel('Decile')
axes[1, 1].set_ylabel('Average History ($)')
axes[1, 1].set_title('Historical Spending by Uplift Decile')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb08/nb08_uplift_by_decile.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance: which features drive differential response?
# Use random forest for better feature importance
from sklearn.ensemble import RandomForestClassifier

# Train RF models separately (T-Learner)
rf_control = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_control.fit(X_control, y_control)

rf_treatment = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_treatment.fit(X_treatment, y_treatment)

# Calculate difference in feature importance
importance_control = rf_control.feature_importances_
importance_treatment = rf_treatment.feature_importances_
importance_diff = importance_treatment - importance_control

feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Control Importance': importance_control,
    'Treatment Importance': importance_treatment,
    'Difference': importance_diff
}).sort_values('Difference', ascending=True)

print("\n=== Feature Importance for Uplift ===")
print(feature_importance_df)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
colors_fi = ['red' if x < 0 else 'green' for x in feature_importance_df['Difference']]
ax.barh(feature_importance_df['Feature'], feature_importance_df['Difference'], color=colors_fi, alpha=0.7, edgecolor='black')
ax.axvline(0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Feature Importance Difference (Treatment - Control)', fontsize=11)
ax.set_title('Which Features Drive Heterogeneous Treatment Response?', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/outputs/nb08/nb08_uplift_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save key results
uplift_results = df_model[['conversion', 'treatment', 'spend', 'recency', 'history', 
                             'uplift_tlearner', 'uplift_slearner', 'uplift_transformed']].copy()
uplift_results.to_csv('../data/outputs/nb08/nb08_uplift_results.csv', index=False)

# Create summary table
summary_uplift = pd.DataFrame({
    'Metric': [
        'Average Predicted Uplift (T-Learner)',
        'Std Dev Uplift',
        '% Customers with Positive Uplift',
        'Max Predicted Uplift',
        'Min Predicted Uplift',
        'AUUC (T-Learner)',
        'Optimal Targeting Percentage',
        'Expected Incremental Conversions (Optimal)'
    ],
    'Value': [
        f"{df_model['uplift_tlearner'].mean():.4f}",
        f"{df_model['uplift_tlearner'].std():.4f}",
        f"{(df_model['uplift_tlearner'] > 0).mean():.1%}",
        f"{df_model['uplift_tlearner'].max():.4f}",
        f"{df_model['uplift_tlearner'].min():.4f}",
        f"{auuc_t:.4f}",
        f"{optimal_pct}%",
        f"{incremental_conversions[optimal_pct-1]:.0f}"
    ]
})

print("\n=== UPLIFT MODELING SUMMARY ===")
print(summary_uplift.to_string(index=False))

summary_uplift.to_csv('../data/outputs/nb08/nb08_uplift_summary.csv', index=False)
print("\nResults saved to: ../data/outputs/nb08/nb08_uplift_results.csv")
print("Summary saved to: ../data/outputs/nb08/nb08_uplift_summary.csv")

## Key Takeaways

1. **Not Everyone Responds the Same**: Uplift modeling reveals heterogeneous treatment effects
2. **Three Methods Compared**:
   - T-Learner: Train separate models (most flexible)
   - S-Learner: Single model with treatment feature (simpler)
   - Class Variable Transformation: Specialized for classification
3. **Qini Curve**: Shows how much value we gain by targeting high-uplift customers
4. **Targeting Strategy**: Use uplift scores to identify which customers to email for maximum ROI
5. **Feature Analysis**: Understand which customer characteristics drive differential responses

## When to Use Uplift Modeling

- **Personalized campaigns**: Target only customers likely to respond
- **Cost optimization**: Send emails only when ROI is positive
- **Segmentation**: Create targeted messaging for different customer types
- **Campaign design**: Choose channels/timing based on predicted uplift

## Limitations

- Requires sufficient sample size in both treatment and control
- Can overfit if not careful with feature selection
- Assumes no unmeasured confounders (treatment is truly random)
- Predictions are noisier than ATE estimates